# Module 06 — Deep Ensemble Training (Cycle 0, Synthetic Data)

**Purpose:** Train a 5-member deep ensemble regressor on synthetic RBP×receptor data as a Cycle 0 placeholder until real ELISA data arrives (~2026-06-01).

**目的：** 在合成 RBP×receptor 数据上训练 5 成员深度集成回归器，作为 Cycle 0 占位符，直到真实 ELISA 数据到来（约 2026-06-01）。

---

## Method: Deep Ensembles

We use **Deep Ensembles** (Lakshminarayanan et al., 2017, *NeurIPS*) — independently train N MLPs, each with a different random seed. At inference:
- **Predicted score** = mean of member outputs
- **Uncertainty** = std of member outputs (epistemic) + member-predicted sigma (aleatoric)

我们使用**深度集成**（Lakshminarayanan et al., 2017, *NeurIPS*）——用不同随机种子独立训练 N 个 MLP。推理时：
- **预测得分** = 成员输出的均值
- **不确定性** = 成员输出的 std（认知不确定性）+ 成员自预测 sigma（偶然不确定性）

### Why Deep Ensembles instead of MC Dropout or GP?

1. **Better calibration than MC Dropout** — Beluch et al. (2018 *CVPR*); Ovadia et al. (2019 *NeurIPS*): ensembles produce better-calibrated uncertainty under distribution shift.
2. **Better scaling than GP** — GP scales O(N³) in training data; at 1280-dim inputs (ESM-2), exact GP is intractable. Deep ensembles scale linearly.
3. **Validated for protein engineering** — Greenman et al. (2025 *NAR Genom Bioinform*) benchmark: deep ensembles are among the top UQ methods for protein function prediction.
4. **Used in ALDE** — Yang et al. (2025 *Nat Commun*) use deep ensembles as the backbone for active-learning-directed evolution; our workflow mirrors this.

**为什么选深度集成而非 MC Dropout 或 GP？**

1. **比 MC Dropout 校准更好** — Beluch et al. (2018 *CVPR*); Ovadia et al. (2019 *NeurIPS*)。
2. **比 GP 更可扩展** — GP 训练复杂度 O(N³)；在 1280 维 ESM-2 输入上，精确 GP 不可行。深度集成线性扩展。
3. **蛋白质工程验证** — Greenman et al. (2025 *NAR Genom Bioinform*) benchmark 中，深度集成是蛋白质功能预测的最佳 UQ 方法之一。
4. **ALDE 中使用** — Yang et al. (2025 *Nat Commun*) 用深度集成作为主动学习定向进化的主干；我们的工作流程与之一致。

---

**References / 参考文献:**
- Lakshminarayanan et al. (2017) "Simple and Scalable Predictive Uncertainty Estimation Using Deep Ensembles." NeurIPS. arXiv:1612.01474.
- Greenman et al. (2025) "Benchmarking uncertainty quantification methods for protein engineering." *NAR Genom Bioinform*.
- Yang et al. (2025) "Active Learning-assisted Directed Evolution (ALDE)." *Nat Commun*.
- Beluch et al. (2018) "The Power of Ensembles for Active Learning." *CVPR*.
- Ovadia et al. (2019) "Can You Trust Your Model's Uncertainty?" *NeurIPS*.

In [ ]:
# Cell 2: Imports, paths, and version printouts
# 第 2 单元：导入、路径和版本输出

import sys
import random
import json
import datetime
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')  # non-interactive / 非交互
import matplotlib.pyplot as plt

# Anchor paths / 锚定路径
# Notebooks live in processes/, so .parents[1] = 06_uncertainty_model/
# Notebook 在 processes/ 里，.parents[1] = 06_uncertainty_model/
NOTEBOOK_DIR = Path.cwd().resolve()
MODULE_DIR   = NOTEBOOK_DIR  # if running from processes/; adjust if needed
# If notebook is in processes/, parent is module dir
if NOTEBOOK_DIR.name == 'processes':
    MODULE_DIR = NOTEBOOK_DIR.parent
elif NOTEBOOK_DIR.name != '06_uncertainty_model':
    # Try to find 06_uncertainty_model in parents
    for p in NOTEBOOK_DIR.parents:
        if p.name == '06_uncertainty_model':
            MODULE_DIR = p
            break

PROCESSES_DIR = MODULE_DIR / 'processes'
OUTPUTS_DIR   = MODULE_DIR / 'outputs' / 'cycle_0'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Add processes/ to sys.path so we can import ensemble.py etc.
# 将 processes/ 加入 sys.path 以便导入 ensemble.py 等模块
if str(PROCESSES_DIR) not in sys.path:
    sys.path.insert(0, str(PROCESSES_DIR))

from ensemble import DeepEnsemble, train_member
from synthetic_data import generate_synthetic_dataset

# Reproducibility seeds
# 可复现性种子（注意：每个 ensemble 成员会用自己的 seed）
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

# Version printout (required by INTERFACE.md §Reproducibility)
# 版本输出（INTERFACE.md §Reproducibility 要求）
print('=== Library versions ===')
print(f'Python:     {sys.version}')
print(f'PyTorch:    {torch.__version__}')
print(f'NumPy:      {np.__version__}')
print(f'Pandas:     {pd.__version__}')
print(f'Matplotlib: {matplotlib.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

def _get_repo_commit():
    try:
        return subprocess.run(
            ['git', 'rev-parse', '--short', 'HEAD'],
            capture_output=True, text=True, cwd=MODULE_DIR
        ).stdout.strip() or 'unknown'
    except Exception:
        return 'unknown'

REPO_SHA = _get_repo_commit()
print(f'Repo commit: {REPO_SHA}')
print(f'Outputs dir: {OUTPUTS_DIR}')

## Step 1: Synthetic Data Generation

**Upstream inputs (not yet available):**
- `04_protein_embedding/outputs/embeddings_*.npz` — RBP and receptor ESM-2 embeddings
- `05_structure_prediction/outputs/affinity_priors.csv` — Boltz-2 ΔG predictions

**Fallback (tonight):** Random Gaussian embeddings (20 RBPs × 4 receptors × 320 dim). Labels ~ N(5.0, 0.5) mimicking typical pKd range.

**上游输入（暂未可用）：**
- `04_protein_embedding/outputs/embeddings_*.npz` — RBP 和受体的 ESM-2 embedding
- `05_structure_prediction/outputs/affinity_priors.csv` — Boltz-2 ΔG 预测

**今晚的 Fallback：** 随机高斯 embedding（20 RBP × 4 受体 × 320 维）。标签 ~ N(5.0, 0.5) 模拟典型 pKd 范围。

In [ ]:
# Cell 3: Generate synthetic dataset
# 第 3 单元：生成合成数据集

EMBED_DIM  = 320   # ESM-2 t6_8M pooled dim / ESM-2 t6_8M 池化维度
N_RBP      = 20    # synthetic RBP count / 合成 RBP 数量
N_RECEPTOR = 4     # synthetic receptor count / 合成受体数量
NOISE_STD  = 0.5   # Gaussian noise on labels / 标签高斯噪声

with warnings.catch_warnings(record=True) as w_list:
    warnings.simplefilter('always')
    X, y, data_meta = generate_synthetic_dataset(
        n_rbp=N_RBP,
        n_receptor=N_RECEPTOR,
        embed_dim=EMBED_DIM,
        noise_std=NOISE_STD,
        seed=GLOBAL_SEED,
    )

if w_list:
    for w in w_list:
        print(f'[WARNING] {w.message}')

print(f'Dataset shape: X={X.shape}, y={y.shape}')
print(f'y stats: mean={y.mean():.3f}, std={y.std():.3f}, min={y.min():.3f}, max={y.max():.3f}')
print(data_meta.head(8).to_string(index=False))

In [ ]:
# Cell 4: Train/val split (80/20 stratified by RBP id)
# 第 4 单元：训练/验证集拆分（80/20，按 RBP id 分层）

rng = np.random.default_rng(GLOBAL_SEED)
idx = rng.permutation(len(X))
split = int(0.8 * len(X))
train_idx, val_idx = idx[:split], idx[split:]

X_train, y_train = X[train_idx], y[train_idx]
X_val,   y_val   = X[val_idx],   y[val_idx]
train_meta = data_meta.iloc[train_idx].reset_index(drop=True)
val_meta   = data_meta.iloc[val_idx].reset_index(drop=True)

print(f'Train: {len(X_train)} pairs | Val: {len(X_val)} pairs')
print(f'Input dim: {X_train.shape[1]} (2 × {EMBED_DIM} = concat[rbp_emb, rec_emb])')

## Step 2: Architecture — EnsembleMember MLP

Each member is a **3-layer MLP** with:
- `input_dim → 256 → 256 → 128 → (mean, log_sigma)`
- `log_sigma` clamped to `[-7, 7]` for numerical stability (Lakshminarayanan 2017 §3.1)
- Training loss: Gaussian NLL = −log p(y | mean, sigma)

每个成员是一个 **3 层 MLP**：
- `input_dim → 256 → 256 → 128 → (均值, 对数标准差)`
- `log_sigma` 截断到 `[-7, 7]` 防止数值不稳定（Lakshminarayanan 2017 §3.1）
- 训练损失：高斯 NLL = −log p(y | mean, sigma)

In [ ]:
# Cell 5: Define and train the 5-member Deep Ensemble
# 第 5 单元：定义并训练 5 成员深度集成

HIDDEN_DIM = 256    # per AGENT_TODO §3.5 / 按 AGENT_TODO §3.5
N_MEMBERS  = 5      # per AGENT_TODO / 按 AGENT_TODO
MAX_EPOCHS = 200    # early stopping saves us / 早停会提前结束
PATIENCE   = 10     # val NLL patience / 验证 NLL 早停耐心
LR         = 1e-4
BATCH_SIZE = 32

input_dim = X_train.shape[1]  # 640
print(f'Architecture: {input_dim} → {HIDDEN_DIM} → {HIDDEN_DIM} → {HIDDEN_DIM//2} → (mean, sigma)')
print(f'Training {N_MEMBERS} members (seeds 0..{N_MEMBERS-1})…')

ens = DeepEnsemble(input_dim=input_dim, hidden_dim=HIDDEN_DIM, n_members=N_MEMBERS)
ens.train(
    X_train, y_train, X_val, y_val,
    max_epochs=MAX_EPOCHS,
    lr=LR,
    patience=PATIENCE,
    batch_size=BATCH_SIZE,
)

print(f'\nTraining complete. Train time: {ens.meta["train_time_s"]}s')
for i, log in enumerate(ens.logs):
    print(f'  Member {i} (seed={log["seed"]}): best_epoch={log["best_epoch"]}, final_val_nll={log["final_val_nll"]:.4f}')

In [ ]:
# Cell 6: Sanity assertions
# 第 6 单元：合理性断言

# Assertion 1: 5 members trained / 断言 1: 5 个成员已训练
assert len(ens.members) == N_MEMBERS, f'Expected {N_MEMBERS} members, got {len(ens.members)}'

# Assertion 2: Predictions are finite / 断言 2: 预测值有限
val_mean, val_std, val_epist = ens.predict(X_val, return_epistemic=True)
assert np.all(np.isfinite(val_mean)), 'Non-finite mean predictions'
assert np.all(np.isfinite(val_std)),  'Non-finite std predictions'

# Assertion 3: ≥90% std > 0 (members disagree somewhere)
# 断言 3: ≥90% std > 0（成员在某处产生分歧）
frac_diverse = float((val_epist > 1e-8).mean())
assert frac_diverse >= 0.90, f'Only {frac_diverse:.1%} of val predictions have std > 0'

print('✓ 5 members trained')
print(f'✓ Predictions finite: mean=[{val_mean.min():.3f}, {val_mean.max():.3f}]')
print(f'✓ Epistemic std range: [{val_epist.min():.4f}, {val_epist.max():.4f}]')
print(f'✓ Diversity: {frac_diverse:.1%} of predictions have std > 0')

In [ ]:
# Cell 7: Save ensemble member state_dicts
# 第 7 单元：保存每个成员的 state_dict

for i, model in enumerate(ens.members):
    path = OUTPUTS_DIR / f'ensemble_member_{i}.pt'
    torch.save(model.state_dict(), path)   # state_dict only, not full model object / 只保存 state_dict
    print(f'Saved: {path.name}  ({path.stat().st_size / 1024:.1f} KB)')

print(f'\nAll {N_MEMBERS} members saved to {OUTPUTS_DIR}')

In [ ]:
# Cell 8: Generate predictions.csv (all pairs) per INTERFACE.md §Module 06
# 第 8 单元：生成 predictions.csv（全部 pair），遵循 INTERFACE.md §Module 06 规范

import hashlib

all_mean, all_std = ens.predict(X)  # predict on all 80 pairs / 在全部 80 个 pair 上预测

model_version = f'{REPO_SHA}_cycle_0'

predictions_df = pd.DataFrame({
    'rbp_id':          data_meta['rbp_id'].tolist(),
    'receptor_id':     data_meta['receptor_id'].tolist(),
    'predicted_score': np.round(all_mean, 6).tolist(),
    'std':             np.round(all_std, 6).tolist(),
    'lower_95':        np.round(all_mean - 1.96 * all_std, 6).tolist(),
    'upper_95':        np.round(all_mean + 1.96 * all_std, 6).tolist(),
    'model_version':   model_version,
})

preds_path = OUTPUTS_DIR / 'predictions.csv'
predictions_df.to_csv(preds_path, index=False)
print(f'Saved predictions.csv ({len(predictions_df)} rows)')
print(predictions_df.head(5).to_string(index=False))

In [ ]:
# Cell 9: Save training_log.json and model_meta.json
# 第 9 单元：保存 training_log.json 和 model_meta.json

# training_log.json — per-member training curves
# training_log.json — 每个成员的训练曲线
log_path = OUTPUTS_DIR / 'training_log.json'
with open(log_path, 'w') as f:
    json.dump(ens.logs, f, indent=2)
print(f'Saved: {log_path.name}')

# model_meta.json — architecture, hyperparams, repo sha, timestamp
# model_meta.json — 架构、超参数、repo sha、时间戳
import sys as _sys
meta_dict = {
    'arch':                'DeepEnsemble_MLP3',
    'hidden_dim':          HIDDEN_DIM,
    'n_members':           N_MEMBERS,
    'input_dim':           input_dim,
    'train_size':          len(X_train),
    'val_size':            len(X_val),
    'max_epochs':          MAX_EPOCHS,
    'lr':                  LR,
    'patience':            PATIENCE,
    'batch_size':          BATCH_SIZE,
    'noise_std':           NOISE_STD,
    'embed_dim_per_protein': EMBED_DIM,
    'data_source':         'synthetic_fallback_random',
    'repo_commit_sha':     REPO_SHA,
    'timestamp_utc':       datetime.datetime.now(datetime.timezone.utc).isoformat().replace('+00:00', 'Z'),
    'train_time_s':        ens.meta.get('train_time_s', -1),
    'torch_version':       torch.__version__,
    'numpy_version':       np.__version__,
    'python_version':      _sys.version.split()[0],
}
meta_path = OUTPUTS_DIR / 'model_meta.json'
with open(meta_path, 'w') as f:
    json.dump(meta_dict, f, indent=2)
print(f'Saved: {meta_path.name}')
print(json.dumps(meta_dict, indent=2))

In [ ]:
# Cell 10: Update MANIFEST.csv
# 第 10 单元：更新 MANIFEST.csv

from run_cycle0 import _update_manifest, _sha256_file
_update_manifest(MODULE_DIR / 'outputs', REPO_SHA)
print('MANIFEST.csv updated')

## Checkpoint Summary

All Cycle 0 artifacts saved. Proceed to `02_calibration_check.ipynb` to evaluate calibration quality.

所有 Cycle 0 产物已保存。继续打开 `02_calibration_check.ipynb` 评估校准质量。

| Artifact | Location |
|---|---|
| Model weights | `outputs/cycle_0/ensemble_member_{0..4}.pt` |
| Predictions | `outputs/cycle_0/predictions.csv` |
| Training log | `outputs/cycle_0/training_log.json` |
| Model metadata | `outputs/cycle_0/model_meta.json` |
| MANIFEST | `outputs/MANIFEST.csv` |